# 11. Idiomas por persona

**Fuente:** `data/raw/idiomaspersonas.csv`  
**Salida:** `data/processed/idiomas_personas.csv`

Nivel de idiomas (lectura, escritura, comprension, conversacion) declarado por cada persona. El archivo de origen duplica IDPERSONA/IDIDIOMA/ULTIMO_CAMBIO/VERSION por un join previo; se colapsan antes de limpiar.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import _preprocesamiento_comun as pc

pd.set_option('display.max_columns', 100)

## 1. Carga de datos crudos

In [ ]:
df = pc.leer_csv('idiomaspersonas.csv', low_memory=False)
df.head()

## 2. Exploración inicial

In [ ]:
pc.resumen(df, 'idiomas_personas')

In [ ]:
df.dtypes

## 3. Limpieza

In [ ]:
df = pc.limpiar_strings(df)
df = pc.quitar_columnas_duplicadas(df)
df = pc.quitar_columnas_vacias(df, umbral=0.90)
df = pc.quitar_columnas_constantes(df)
df = df.drop(columns=['ULTIMO_CAMBIO', 'REFARCHIVO'], errors='ignore')

## 5. Tipado de fechas e identificadores

In [ ]:
df = pc.castear_fechas(df, ['ULTIMO_CAMBIO', 'FECHASUBIDAARCHIVO'])
df = pc.castear_enteros(df, ['IDPERSONA', 'IDIDIOMA', 'IDIDIOMAPERSONA'])
df = pc.convertir_sn_a_binario(df, ['LENGUANATIVA'])

## Gráficos exploratorios

Vistas rápidas para apoyar la construcción del catálogo de variables del perfil (Fase 1-2 de la metodología): estacionalidad/tendencia temporal, categorías dominantes y forma de la distribución de las variables numéricas.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 4)

**Idiomas más declarados** por el personal.

In [ ]:
pc.grafico_barras(df['IDIOMA'], 'Idiomas más declarados', top=12)

**Nivel MCER declarado** (cuando está disponible).

In [ ]:
pc.grafico_barras(df['NIVELMCER'], 'Distribución de NIVELMCER', top=6, horizontal=False)

**Nivel de lectura declarado**.

In [ ]:
pc.grafico_barras(df['NIVELLECTURA'], 'Distribución de NIVELLECTURA', horizontal=False)

**Lengua nativa** (LENGUANATIVA, ya convertida a binario).

In [ ]:
conteo_lengua = df['LENGUANATIVA'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(4, 4))
conteo_lengua.plot(kind='bar', ax=ax, color=pc.COLOR_PRINCIPAL)
ax.set_title('LENGUANATIVA (0 = No, 1 = Sí)')
plt.tight_layout()

## 7. Verificación final

In [ ]:
pc.resumen(df, 'idiomas_personas (procesado)')
df.head()

## 8. Guardado en data/processed

In [ ]:
pc.guardar_procesado(df, 'idiomas_personas.csv')